# Variações dos Datasets para Experimentos (a partir do Base com Agro Lags)

**IMPORTANTE - ordem de execução:**
1. `indicadores_agro/tratar_indicadores_agro.ipynb`
2. `tratamento_datasets.ipynb`
3. `variacoes_datasets.ipynb`

Este notebook parte dos datasets em `datasets_base` (já contendo os lags de fechamento dos indicadores agro de soja e câmbio) e cria 3 variações adicionais para experimentos:
1. **datasets_janelas** (Base + Lags de mercado)
2. **datasets_dummy** (Base + Variável Dummy de período)
3. **datasets_indicadores** (Base + Indicadores de Mercado)

Ao final, os 4 folders usados nos experimentos ficam:
- `datasets_base`
- `datasets_dummy`
- `datasets_indicadores`
- `datasets_janelas`

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

print('Bibliotecas carregadas com sucesso.')

## 1. Configuração de pastas

In [ ]:
BASE_DIR = Path('..').resolve()
DATASETS_DIR = BASE_DIR / 'datasets'

# Pastas de entrada (datasets base já com agro lags, gerados no tratamento_datasets.ipynb)
INPUT_REGRESSAO_DIR = DATASETS_DIR / 'datasets_base' / 'regressao'
INPUT_CLASSIFICACAO_DIR = DATASETS_DIR / 'datasets_base' / 'classificacao'

# Pastas de saída das variações
OUTPUT_JANELAS_REG = DATASETS_DIR / 'datasets_janelas' / 'regressao'
OUTPUT_JANELAS_CLF = DATASETS_DIR / 'datasets_janelas' / 'classificacao'

OUTPUT_DUMMY_REG = DATASETS_DIR / 'datasets_dummy' / 'regressao'
OUTPUT_DUMMY_CLF = DATASETS_DIR / 'datasets_dummy' / 'classificacao'

OUTPUT_INDICADORES_REG = DATASETS_DIR / 'datasets_indicadores' / 'regressao'
OUTPUT_INDICADORES_CLF = DATASETS_DIR / 'datasets_indicadores' / 'classificacao'

# Criar todas as pastas de saída
for folder in [
    OUTPUT_JANELAS_REG,
    OUTPUT_JANELAS_CLF,
    OUTPUT_DUMMY_REG,
    OUTPUT_DUMMY_CLF,
    OUTPUT_INDICADORES_REG,
    OUTPUT_INDICADORES_CLF,
]:
    folder.mkdir(parents=True, exist_ok=True)

print('Pastas de entrada:')
print('  - Regressão:', INPUT_REGRESSAO_DIR)
print('  - Classificação:', INPUT_CLASSIFICACAO_DIR)
print('\nPastas de saída criadas:')
print('  - datasets_janelas:', OUTPUT_JANELAS_REG.parent)
print('  - datasets_dummy:', OUTPUT_DUMMY_REG.parent)
print('  - datasets_indicadores:', OUTPUT_INDICADORES_REG.parent)

In [ ]:
# Verificar se os datasets base existem
csv_reg = sorted(INPUT_REGRESSAO_DIR.glob('*_tratado.csv'))
csv_clf = sorted(INPUT_CLASSIFICACAO_DIR.glob('*_tratado.csv'))

if not csv_reg or not csv_clf:
    raise FileNotFoundError(
        'Datasets base não encontrados! Execute primeiro o notebook tratamento_datasets.ipynb'
    )

print(f'Datasets base encontrados: {len(csv_reg)} para regressão, {len(csv_clf)} para classificação')
print('OBS: Esses arquivos já devem conter os lags de fechamento dos indicadores agro (soja e câmbio).')

print('\nArquivos de regressão:')
for f in csv_reg:
    print('  -', f.name)
print('\nArquivos de classificação:')
for f in csv_clf:
    print('  -', f.name)

## 2. Funções de transformação
### 2.1 Utilitário de carregamento (base já com agro lags)

In [ ]:
def carregar_dataset_base(file_path: Path) -> pd.DataFrame:
    """
    Carrega dataset base já tratado (incluindo agro lags) preservando índice temporal.
    """
    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
    df = df.sort_index()
    return df


exemplo = carregar_dataset_base(csv_reg[0]) if csv_reg else None
if exemplo is not None:
    agro_cols = [c for c in exemplo.columns if c.startswith('agro_')]
    print('Exemplo carregado:', exemplo.shape)
    print('Total de colunas agro no base:', len(agro_cols))
    print('Primeiras colunas agro:', agro_cols[:8])

### 2.2 Janelas Deslizantes (Lag Features)

In [ ]:
def adicionar_janelas_deslizantes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adiciona janelas deslizantes para trás (2, 4 e 6 dias) para Open, High, Low e Close.
    Gera 12 novas colunas (4 features × 3 períodos).
    """
    df = df.copy()
    features = ['Open', 'High', 'Low', 'Close']
    lags = [2, 4, 6]
    
    for feature in features:
        if feature in df.columns:
            for lag in lags:
                col_name = f'{feature}_lag{lag}d'
                df[col_name] = df[feature].shift(lag)
    
    # Remove linhas com NaN geradas pelos shifts
    df = df.dropna()
    
    return df

print('Função adicionar_janelas_deslizantes() definida.')

### 2.3 Variável Dummy de Período

In [ ]:
def adicionar_variavel_dummy(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adiciona variável dummy de período baseada em datas específicas.
    - 0: antes de 2021-05-05 (período inicial)
    - 1: entre 2021-05-05 e 2022-02-07 (período de alta volatilidade)
    - 2: após 2022-02-07 (período de estabilização)
    
    Aplica One-Hot Encoding (drop='first') gerando period_1 e period_2.
    """
    df = df.copy()
    
    # Garantir que o índice é datetime
    if not isinstance(df.index, pd.DatetimeIndex):
        if 'Date' in df.columns:
            df['Date'] = pd.to_datetime(df['Date'])
            df = df.set_index('Date')
        else:
            df.index = pd.to_datetime(df.index)
    
    # Criar variável dummy categórica
    df['dummy_period'] = 0
    df.loc[df.index >= pd.to_datetime('2021-05-05'), 'dummy_period'] = 1
    df.loc[df.index >= pd.to_datetime('2022-02-07'), 'dummy_period'] = 2
    
    # Aplicar One-Hot Encoding (drop='first' para evitar multicolinearidade)
    dummy_encoded = pd.get_dummies(df['dummy_period'], prefix='period', drop_first=True, dtype=int)
    
    # Concatenar com DataFrame original e remover coluna dummy_period original
    df = pd.concat([df, dummy_encoded], axis=1)
    df = df.drop(columns=['dummy_period'])
    
    return df

print('Função adicionar_variavel_dummy() definida.')

### 2.4 Indicadores de Mercado

In [ ]:
def calcular_obv(close: pd.Series, volume: pd.Series) -> pd.Series:
    """
    On-Balance Volume (OBV): indicador de momentum baseado em volume.
    """
    direction = np.where(close > close.shift(1), 1, np.where(close < close.shift(1), -1, 0))
    obv = (direction * volume).cumsum()
    return pd.Series(obv, index=close.index, name='OBV')


def calcular_fwma(series: pd.Series, period: int = 14) -> pd.Series:
    """
    Fibonacci Weighted Moving Average (FWMA).
    Usa pesos baseados na sequência de Fibonacci.
    """
    # Gerar pesos de Fibonacci
    fib = [1, 1]
    for i in range(2, period):
        fib.append(fib[i-1] + fib[i-2])
    weights = np.array(fib[:period], dtype=float)
    weights = weights / weights.sum()
    
    # Calcular média ponderada
    fwma = series.rolling(window=period).apply(lambda x: np.dot(x, weights), raw=True)
    return pd.Series(fwma, index=series.index, name='FWMA')


def calcular_tema(series: pd.Series, period: int = 14) -> pd.Series:
    """
    Triple Exponential Moving Average (TEMA).
    TEMA = 3*EMA1 - 3*EMA2 + EMA3
    """
    ema1 = series.ewm(span=period, adjust=False).mean()
    ema2 = ema1.ewm(span=period, adjust=False).mean()
    ema3 = ema2.ewm(span=period, adjust=False).mean()
    tema = 3 * ema1 - 3 * ema2 + ema3
    return pd.Series(tema, index=series.index, name='TEMA')


def calcular_hlc3(high: pd.Series, low: pd.Series, close: pd.Series) -> pd.Series:
    """
    HLC3 (Typical Price): média de High, Low e Close.
    """
    hlc3 = (high + low + close) / 3
    return pd.Series(hlc3, index=high.index, name='HLC3')


def calcular_bollinger_bands(series: pd.Series, period: int = 20, std_dev: float = 2.0) -> pd.DataFrame:
    """
    Bollinger Bands: banda superior, média móvel e banda inferior.
    """
    sma = series.rolling(window=period).mean()
    std = series.rolling(window=period).std()
    
    upper_band = sma + (std_dev * std)
    lower_band = sma - (std_dev * std)
    
    return pd.DataFrame({
        'BB_upper': upper_band,
        'BB_middle': sma,
        'BB_lower': lower_band
    }, index=series.index)


print('Funções de indicadores técnicos definidas:')
print('  - calcular_obv()')
print('  - calcular_fwma()')
print('  - calcular_tema()')
print('  - calcular_hlc3()')
print('  - calcular_bollinger_bands()')

In [ ]:
def adicionar_indicadores_tecnicos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adiciona todos os indicadores técnicos ao DataFrame:
    - OBV (On-Balance Volume)
    - FWMA (Fibonacci Weighted Moving Average)
    - TEMA (Triple Exponential Moving Average)
    - HLC3 (Typical Price)
    - Bollinger Bands (upper, middle, lower)
    """
    df = df.copy()
    
    # Verificar colunas necessárias
    required = ['Open', 'High', 'Low', 'Close', 'Volume']
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f'Colunas ausentes para indicadores técnicos: {missing}')
    
    # OBV
    df['OBV'] = calcular_obv(df['Close'], df['Volume'])
    
    # FWMA (usando Close)
    df['FWMA'] = calcular_fwma(df['Close'], period=14)
    
    # TEMA (usando Close)
    df['TEMA'] = calcular_tema(df['Close'], period=14)
    
    # HLC3
    df['HLC3'] = calcular_hlc3(df['High'], df['Low'], df['Close'])
    
    # Bollinger Bands
    bb = calcular_bollinger_bands(df['Close'], period=20, std_dev=2.0)
    df = pd.concat([df, bb], axis=1)
    
    # Remove linhas com NaN geradas pelos cálculos
    df = df.dropna()
    
    return df

print('Função adicionar_indicadores_tecnicos() definida.')

## 3. Processamento em lote
### 3.1 Datasets com Lags (a partir do Base com Agro Lags)

In [ ]:
print('=' * 80)
print('BASE DE ENTRADA')
print('=' * 80)
print('datasets_base já contém os lags de fechamento dos indicadores agro (soja e câmbio).')
print('Este notebook gera apenas: datasets_janelas, datasets_dummy e datasets_indicadores.')

### 3.2 Datasets com Lags (datasets_janelas)

In [ ]:
print('=' * 80)
print('VARIAÇÃO 1: DATASETS_JANELAS (BASE + LAGS)')
print('=' * 80)

# Processar arquivos de regressão
print('\n--- Processando datasets de REGRESSÃO ---')
for file_path in csv_reg:
    print(f'\nProcessando: {file_path.name}')
    df_base = carregar_dataset_base(file_path)
    print(f'  Shape base: {df_base.shape}')
    
    df_janelas = adicionar_janelas_deslizantes(df_base)
    print(f'  Shape com lags: {df_janelas.shape}')
    
    output_path = OUTPUT_JANELAS_REG / file_path.name
    df_janelas.to_csv(output_path, index=True)
    print(f'  Salvo em: {output_path}')

# Processar arquivos de classificação
print('\n--- Processando datasets de CLASSIFICAÇÃO ---')
for file_path in csv_clf:
    print(f'\nProcessando: {file_path.name}')
    df_base = carregar_dataset_base(file_path)
    print(f'  Shape base: {df_base.shape}')
    
    df_janelas = adicionar_janelas_deslizantes(df_base)
    print(f'  Shape com lags: {df_janelas.shape}')
    
    output_path = OUTPUT_JANELAS_CLF / file_path.name
    df_janelas.to_csv(output_path, index=True)
    print(f'  Salvo em: {output_path}')

print('\n✓ datasets_janelas gerados com sucesso!')

### 3.3 Datasets com Variável Dummy (datasets_dummy)

In [ ]:
print('=' * 80)
print('VARIAÇÃO 2: DATASETS_DUMMY (BASE + VARIÁVEL DUMMY)')
print('=' * 80)

# Processar arquivos de regressão
print('\n--- Processando datasets de REGRESSÃO ---')
for file_path in csv_reg:
    print(f'\nProcessando: {file_path.name}')
    df_base = carregar_dataset_base(file_path)
    print(f'  Shape base: {df_base.shape}')
    
    df_dummy = adicionar_variavel_dummy(df_base)
    print(f'  Shape com dummy: {df_dummy.shape}')
    
    output_path = OUTPUT_DUMMY_REG / file_path.name
    df_dummy.to_csv(output_path, index=True)
    print(f'  Salvo em: {output_path}')

# Processar arquivos de classificação
print('\n--- Processando datasets de CLASSIFICAÇÃO ---')
for file_path in csv_clf:
    print(f'\nProcessando: {file_path.name}')
    df_base = carregar_dataset_base(file_path)
    print(f'  Shape base: {df_base.shape}')
    
    df_dummy = adicionar_variavel_dummy(df_base)
    print(f'  Shape com dummy: {df_dummy.shape}')
    
    output_path = OUTPUT_DUMMY_CLF / file_path.name
    df_dummy.to_csv(output_path, index=True)
    print(f'  Salvo em: {output_path}')

print('\n✓ datasets_dummy gerados com sucesso!')

### 3.4 Datasets com Indicadores de Mercado (datasets_indicadores)

In [ ]:
print('=' * 80)
print('VARIAÇÃO 3: DATASETS_INDICADORES (BASE + INDICADORES DE MERCADO)')
print('=' * 80)

# Processar arquivos de regressão
print('\n--- Processando datasets de REGRESSÃO ---')
for file_path in csv_reg:
    print(f'\nProcessando: {file_path.name}')
    df_base = carregar_dataset_base(file_path)
    print(f'  Shape base: {df_base.shape}')
    
    df_indicadores = adicionar_indicadores_tecnicos(df_base)
    print(f'  Shape com indicadores de mercado: {df_indicadores.shape}')
    
    output_path = OUTPUT_INDICADORES_REG / file_path.name
    df_indicadores.to_csv(output_path, index=True)
    print(f'  Salvo em: {output_path}')

# Processar arquivos de classificação
print('\n--- Processando datasets de CLASSIFICAÇÃO ---')
for file_path in csv_clf:
    print(f'\nProcessando: {file_path.name}')
    df_base = carregar_dataset_base(file_path)
    print(f'  Shape base: {df_base.shape}')
    
    df_indicadores = adicionar_indicadores_tecnicos(df_base)
    print(f'  Shape com indicadores de mercado: {df_indicadores.shape}')
    
    output_path = OUTPUT_INDICADORES_CLF / file_path.name
    df_indicadores.to_csv(output_path, index=True)
    print(f'  Salvo em: {output_path}')

print('\n✓ datasets_indicadores gerados com sucesso!')

## 4. Resumo final

In [ ]:
print('=' * 80)
print('RESUMO FINAL')
print('=' * 80)

# Contar arquivos gerados
janelas_reg = sorted(OUTPUT_JANELAS_REG.glob('*.csv'))
janelas_clf = sorted(OUTPUT_JANELAS_CLF.glob('*.csv'))
dummy_reg = sorted(OUTPUT_DUMMY_REG.glob('*.csv'))
dummy_clf = sorted(OUTPUT_DUMMY_CLF.glob('*.csv'))
indicadores_reg = sorted(OUTPUT_INDICADORES_REG.glob('*.csv'))
indicadores_clf = sorted(OUTPUT_INDICADORES_CLF.glob('*.csv'))

print('\nPASTA datasets_base/ (gerada no notebook anterior, já com agro lags):')
print(f'  Regressão: {len(csv_reg)} arquivos')
for f in csv_reg:
    print(f'    - {f.name}')
print(f'  Classificação: {len(csv_clf)} arquivos')
for f in csv_clf:
    print(f'    - {f.name}')

print('\nPASTA datasets_janelas/ (BASE + LAGS):')
print(f'  Regressão: {len(janelas_reg)} arquivos')
for f in janelas_reg:
    print(f'    - {f.name}')
print(f'  Classificação: {len(janelas_clf)} arquivos')
for f in janelas_clf:
    print(f'    - {f.name}')

print('\nPASTA datasets_dummy/ (BASE + DUMMY):')
print(f'  Regressão: {len(dummy_reg)} arquivos')
for f in dummy_reg:
    print(f'    - {f.name}')
print(f'  Classificação: {len(dummy_clf)} arquivos')
for f in dummy_clf:
    print(f'    - {f.name}')

print('\nPASTA datasets_indicadores/ (BASE + INDICADORES DE MERCADO):')
print(f'  Regressão: {len(indicadores_reg)} arquivos')
for f in indicadores_reg:
    print(f'    - {f.name}')
print(f'  Classificação: {len(indicadores_clf)} arquivos')
for f in indicadores_clf:
    print(f'    - {f.name}')

total = (
    len(csv_reg) + len(csv_clf) +
    len(janelas_reg) + len(janelas_clf) +
    len(dummy_reg) + len(dummy_clf) +
    len(indicadores_reg) + len(indicadores_clf)
)
print(f'\nTotal de {total} datasets disponíveis nas 4 pastas de experimento!')

## 5. Visualização das features adicionadas

In [ ]:
# Carregar um exemplo de cada variação para mostrar as colunas
print('=' * 80)
print('COLUNAS NAS PASTAS DE EXPERIMENTO')
print('=' * 80)

if csv_reg:
    df_exemplo = pd.read_csv(csv_reg[0], index_col=0, nrows=5)
    print(f'\nDATASETS_BASE ({csv_reg[0].name}):')
    print(f'   Colunas: {list(df_exemplo.columns)}')

if janelas_reg:
    df_exemplo = pd.read_csv(janelas_reg[0], index_col=0, nrows=5)
    print(f'\nDATASETS_JANELAS ({janelas_reg[0].name}):')
    print(f'   Colunas: {list(df_exemplo.columns)}')

if dummy_reg:
    df_exemplo = pd.read_csv(dummy_reg[0], index_col=0, nrows=5)
    print(f'\nDATASETS_DUMMY ({dummy_reg[0].name}):')
    print(f'   Colunas: {list(df_exemplo.columns)}')

if indicadores_reg:
    df_exemplo = pd.read_csv(indicadores_reg[0], index_col=0, nrows=5)
    print(f'\nDATASETS_INDICADORES ({indicadores_reg[0].name}):')
    print(f'   Colunas: {list(df_exemplo.columns)}')